# Continued Pre-Training (CPT) of LLM with PyTorch FSDP and QLora on Amazon SageMaker Training jobs

## Lab 3 - LLM Deployment

In this notebook, we are going to deploy the fine-tuned LLM using SageMaker Real-time endpoint

***

### Prerequistes

#### Setup and dependencies

In [ ]:
import boto3
import os
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

model_id = "Qwen/Qwen3-4B-Instruct-2507"

model_name = f"{model_id.split('/')[-1].replace('.', '-')}-cpt"
endpoint_config_name = f"{model_id.split('/')[-1].replace('.', '-')}-cpt-config"
endpoint_name = f"{model_id.split('/')[-1].replace('.', '-')}-cpt-endpoint"
ic_name = f"{model_id.split('/')[-1].replace('.', '-')}-cpt-ic"

***

## Load Fine-Tuned model

Note: Run `train_fn` with `merge_weights=True` for merging the trained adapter

In [ ]:
job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-cpt"

In [ ]:
def get_last_job_name(job_name_prefix):
    matching_jobs = []
    next_token = None

    while True:
        # Prepare the search parameters
        search_params = {
            "Resource": "TrainingJob",
            "SearchExpression": {
                "Filters": [
                    {
                        "Name": "TrainingJobName",
                        "Operator": "Contains",
                        "Value": job_name_prefix,
                    },
                    {
                        "Name": "TrainingJobStatus",
                        "Operator": "Equals",
                        "Value": "Completed",
                    },
                ]
            },
            "SortBy": "CreationTime",
            "SortOrder": "Descending",
            "MaxResults": 100,
        }

        # Add NextToken if we have one
        if next_token:
            search_params["NextToken"] = next_token

        # Make the search request
        search_response = sm_client.search(**search_params)

        # Filter and add matching jobs
        matching_jobs.extend(
            [
                job["TrainingJob"]["TrainingJobName"]
                for job in search_response["Results"]
                if job["TrainingJob"]["TrainingJobName"].startswith(job_name_prefix)
            ]
        )

        # Check if we have more results to fetch
        next_token = search_response.get("NextToken")
        if (
            not next_token or matching_jobs
        ):  # Stop if we found at least one match or no more results
            break

    if not matching_jobs:
        raise ValueError(
            f"No completed training jobs found starting with prefix '{job_name_prefix}'"
        )

    return matching_jobs[0]

In [ ]:
job_name = get_last_job_name(job_prefix)

job_name

***

### Create Endpoint Configuration

Define inference configuration

In [ ]:
instance_count = 1
instance_type = "ml.g5.2xlarge"
number_of_gpu = 1
health_check_timeout = 700

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
            inference_ami_version="al2-ami-sagemaker-inference-gpu-3-1",
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"},
        )
    ],
)

### Create Endpoint

A SageMaker Endpoint is a fully managed, always-on HTTPS API that hosts your deployed model and serves real-time inference requests.

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

### Create Model

Get the image URI

In [ ]:
import json

In [ ]:
region = sess.boto_region_name
CONTAINER_VERSION = "vllm:0.14.0-gpu-py312-cu129-ubuntu22.04-sagemaker"

image_uri = (
    f"763104351884.dkr.ecr.{region}.amazonaws.com/{CONTAINER_VERSION}"
)

image_uri

In [ ]:
env = {
    "SM_VLLM_MODEL": "/opt/ml/model",
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "SM_VLLM_DTYPE": "bfloat16",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.8",
    "SM_VLLM_MAX_MODEL_LEN": json.dumps(1024 * 8),
    "SM_VLLM_MAX_NUM_SEQS": "8",
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=f"s3://{bucket_name}/{job_prefix}/{job_name}/output/model.tar.gz",
                s3_data_type="S3Prefix",
                compression_type="Gzip",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

pprint(fine_tuned_model)

### Create Inference Component

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (
    InferenceComponentSpecification,
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
)

# Step 3: Create InferenceComponent
inference_component = InferenceComponent.create(
    inference_component_name=ic_name,
    endpoint_name=endpoint_name,
    variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=model_name,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=10240,
            number_of_accelerator_devices_required=number_of_gpu,
        ),
    ),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
    region=region,
)

print(f"InferenceComponent created: {inference_component.inference_component_name}")
inference_component.wait_for_status("InService")
print(f"Endpoint {ic_name} is InService")

***

### Test endpoint

In [ ]:
import boto3
import io
import json

In [ ]:
sagemaker_client = boto3.client(service_name="sagemaker-runtime")

### Iterator class for streaming inference

Utility class to parse streaming responses

In [ ]:
class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a single streaming response line and return (content, tool_call_delta)."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None, None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                delta = choice.get("delta", {})
                if "content" in delta and delta["content"]:
                    return delta["content"], None
                if "tool_calls" in delta:
                    return None, delta["tool_calls"]
    except json.JSONDecodeError:
        pass

    return None, None

In [ ]:
prompt = """
"Multiply 3/8 with 7/10"
"""

In [ ]:
request_body = {
    "model_name": ic_name,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stop": ["<|im_end|>"],
    "stream": True,
}

response = sagemaker_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

generated_text = ""
tool_calls = []

for line in LineIterator(response["Body"]):
    if line:
        content, tool_call_delta = parse_streaming_response(line.decode("utf-8"))
        if content:
            generated_text += content
            print(content, end="", flush=True)
        if tool_call_delta:
            for tc in tool_call_delta:
                idx = tc.get("index", 0)
                while len(tool_calls) <= idx:
                    tool_calls.append(
                        {"type": "function", "function": {"name": "", "arguments": ""}}
                    )
                if "function" in tc:
                    if tc["function"].get("name"):
                        tool_calls[idx]["function"]["name"] = tc["function"]["name"]
                    if "arguments" in tc["function"]:
                        tool_calls[idx]["function"]["arguments"] += tc["function"][
                            "arguments"
                        ]

if tool_calls:
    pprint(tool_calls)

***

### Delete resources

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint (optional - if you want to remove the endpoint too)
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config (optional)
EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()